In [ ]:
OPENROUTER_API_KEY = ''

In [ ]:
MODEL = 'anthropic/claude-sonnet-4.6'

In [ ]:
import pandas as pd
import traceback
import time
from openai import OpenAI
from IPython import get_ipython
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth

# --- 1. CONFIGURATION ---
SHEET_NAME = 'Code_Fix_Progress_Batch_Save_Claude_GH-33'
SAVE_INTERVAL = 5  # Sync to Google Sheets every 10 execution attempts
START_ROW = 33      # Adjust this if the script stops again


# --- 2. AUTHENTICATION & INITIALIZATION ---


# Initialize OpenAI Client via OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "http://localhost:3000",
        "X-Title": "Local Research Script",
    }
)

# Setup Google Sheet
try:
    sh = gc.open(SHEET_NAME)
except gspread.exceptions.SpreadsheetNotFound:
    sh = gc.create(SHEET_NAME)
worksheet = sh.sheet1

# --- 3. HELPER FUNCTIONS ---

def sync_to_sheets(dataframe):
    """Utility to push current DF state to Google Sheets."""
    try:
        set_with_dataframe(worksheet, dataframe)
        print(f"📊 [{time.strftime('%H:%M:%S')}] Batch synced to Google Sheets.")
    except Exception as e:
        print(f"⚠️ Sheet Sync Failed: {e}")

def execute_and_verify(code_string):
    """Executes code in Colab and returns (success, error_message)."""
    try:
        result = get_ipython().run_cell(code_string)
        if result.success:
            return True, None
        else:
            return False, str(result.error_in_exec)
    except Exception as e:
        return False, traceback.format_exc()

def get_gpt_fix(buggy_code, error_log):
    """Requests a fix from GPT based on specific constraints."""
    prompt = f""" You are an expert Python Developer. You are given a code base, <Buggy Code>, that generates the error. You are also provided the error log in the <traceback>. Below are the three constraints you need to focus.

Constraint 1: If the fix requires any external libraries or updated packages, provide the necessary pip install commands at the very top of your response.

Constraint 2: Provide the corrected version of the code. If errors are related to API keys or quota exceeded, use mock api call and keys to test the agent compilation.

Constraint 3: Do not provide conversational filler or long explanations. Only provide the pip commands (if needed) and the code block.
<Buggy Code>: {buggy_code}
<traceback>:
{error_log}
Now, write the code and return the code in this format:

<code>
{{your_code_here}}
</code>
"""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content
        if "<code>" in content:
            return content.split("<code>")[1].split("</code>")[0].strip()
        return content
    except Exception as e:
        print(f"OpenRouter API Error: {e}")
        return buggy_code

# --- 4. EXECUTION ENGINE ---

# IMPORTANT: Ensure 'df' is defined in your environment before running this block
df_test = df.copy()

cols_to_process = ['Code1', 'Code2']
call_count = 0

# Initialize columns only if they don't exist (prevents wiping old data on restart)
for col in cols_to_process:
    for p in range(1, 4):
        if f"check{col}@{p}" not in df_test.columns:
            df_test[f"check{col}@{p}"] = ""
        if f"{col}_pass{p}_code" not in df_test.columns:
            df_test[f"{col}_pass{p}_code"] = ""

# Slice the dataframe to resume from the specified row
df_to_process = df_test.iloc[START_ROW:]

print(f"🚀 Starting self-healing loop from row {START_ROW}...")

for index, row in df_to_process.iterrows():
    for col in cols_to_process:
        current_code = row[col]

        # Retry logic (Pass 1 to 3)
        for p in range(1, 4):
            call_count += 1
            check_label = f"check{col}@{p}"
            code_store = f"{col}_pass{p}_code"

            print(f"\n[Call #{call_count}] Row {index} | {col} | Pass {p}")

            success, error_msg = execute_and_verify(current_code)

            if success:
                df_test.at[index, check_label] = "Success"
                df_test.at[index, code_store] = current_code

                if call_count % SAVE_INTERVAL == 0:
                    sync_to_sheets(df_test)

                print(f"✅ Successful compilation.")
                break
            else:
                df_test.at[index, check_label] = f"Failed: {str(error_msg)[:150]}"
                df_test.at[index, code_store] = current_code

                if call_count % SAVE_INTERVAL == 0:
                    sync_to_sheets(df_test)

                print(f"❌ Failed. Requesting fix from AI...")

                if p < 3:
                    current_code = get_gpt_fix(current_code, error_msg)
                else:
                    print(f"⚠️ Max attempts reached for this column.")

# Final sync to save remaining progress
sync_to_sheets(df_test)
print("\n✨ All rows processed. Check your Google Sheet!")

🚀 Starting self-healing loop from row 33...

[Call #1] Row 34 | Code1 | Pass 1


/tmp/ipykernel_35856/3457464495.py:30: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(max_results=3)
/tmp/ipykernel_35856/3457464495.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


TypeError: create_react_agent() got unexpected keyword arguments: {'state_modifier': 'You are a research expert. Use web search to find accurate factual information about historical events, current data, company information, employee counts, and other real-world facts. Provide clear, concise answers based on your research.'}

❌ Failed. Requesting fix from AI...

[Call #2] Row 34 | Code1 | Pass 2

Query: What is 15 multiplied by 7, then add 23 to the result?


/tmp/ipykernel_35856/3267203425.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(
/tmp/ipykernel_35856/3267203425.py:38: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(


ValueError: Unsupported function

{'type': 'object', 'properties': {'next': {'type': 'string', 'enum': ['research_expert', 'math_expert', 'FINISH'], 'description': 'The next worker to act or FINISH if done'}}, 'required': ['next']}

To use a JSON schema as a function, it must have a top-level 'title' key to be used as the function name.

... (192 outputs left out)

TimeoutException: Requesting secret OPENAI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

❌ Failed. Requesting fix from AI...

[Call #135] Row 62 | Code1 | Pass 2

=== AGENTIC ARTIFACTS - FINAL RESULT ===

USER REQUEST: Create a Python web scraper that extracts product information from e-commerce websites

PLAN:
{"artifact_type": "Python Script", "components": ["scraper", "parser", "exporter"], "steps": ["1. Setup", "2. Scrape", "3. Parse", "4. Export"], "output_format": "CSV/JSON"}

OVERVIEW:
Overview:
1. Executive Summary: A robust web scraping solution
2. Key Features: Multi-page scraping, data extraction, export
3. Technical Requirements: Python 3.8+, requests, BeautifulSoup
4. Timeline: 2-4 hours

GENERATED ARTIFACT:
```python
import requests
from bs4 import BeautifulSoup
import json

def scrape_products(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    products = []
    for item in soup.select('.product-item'):
        product = {
            'name': item.se

TimeoutException: Requesting secret OPENAI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

❌ Failed. Requesting fix from AI...

[Call #137] Row 62 | Code2 | Pass 2
PLAN:
Mock Plan: 1. Research topic 2. Outline structure 3. Write content 4. Review and finalize

OVERVIEW:
Mock Overview: This artifact provides a comprehensive guide covering key concepts, implementation details, and best practices.

GENERATED ARTIFACT:
Mock Plan: 1. Research topic 2. Outline structure 3. Write content 4. Review and finalize

PREVIEW:
Mock Plan: 1. Research topic 2. Outline structure 3. Write content 4. Review and finalize

STATUS: generated
✅ Successful compilation.
📊 [20:00:25] Batch synced to Google Sheets.

✨ All rows processed. Check your Google Sheet!
